# 🛠️ SQL 3 – Funktionen für den Gold Layer

## 🎯 Lernziele
Am Ende dieses Labs kannst du:
- Verschachtelte Daten (Arrays) mit **EXPLODE** in Zeilen auflösen
- Mit **QUALIFY** pro Gruppe nur die richtige Zeile behalten – z. B. die neueste Version eines Kunden
- Fehlende Werte mit **COALESCE** ersetzen
- Kategorien und Kennzeichen mit **CASE** und **IFF** ableiten
- Codes aus Quellsystemen mit **DECODE** in lesbare Werte übersetzen
- Mit **MAX_BY** „den Wert, der zum Maximum gehört“ auswählen
- Mit **FILTER (WHERE …)** mehrere bedingte Aggregate in einem Durchgang berechnen
- Mit **DATEADD** und **DATEDIFF** mit Datumswerten rechnen

## Einführung

**Warum dieses Lab?**

Quellsysteme liefern Daten selten so, wie ein Report sie braucht. Bestellungen kommen als verschachteltes JSON, Kunden existieren in mehreren Versionen, Felder sind leer und Statuswerte werden als Ein-Buchstaben-Codes gespeichert. Die Funktionen in diesem Lab sind der Werkzeugkasten, um all das zu beheben – sie stecken in fast jeder Silver- und Gold-Tabelle eines Lakehouse, auch in der Sunny-Bay-Pipeline.

## Anleitung

Bevor du startest, prüfe bitte:
- Du hast **SQL 1** und **SQL 2** abgeschlossen (Lab 0 hat das Schema `sql_sandbox` für deine eigenen Tabellen angelegt).
- Dieses Notebook ist mit **Serverless** Compute oder einem **SQL warehouse** verbunden.

1. Setze den Standard-Katalog erneut (ersetze ihn, falls du in Lab 0 einen anderen Katalog verwendet hast):

In [ ]:
USE CATALOG sunny_bay_germany;

**Schritt 1: Arrays auflösen mit EXPLODE**

Der Onlineshop liefert jede Bestellung als **ein JSON-Dokument** – die Produkte der Bestellung stehen in einem **Array** namens `items`. Reports brauchen aber eine Zeile pro Produkt. `EXPLODE(array)` macht aus jedem Element eines Arrays eine eigene Zeile; die übrigen Spalten werden wiederholt. `POSEXPLODE` liefert zusätzlich die Position des Elements (beginnend bei 0).

1. Führe das Mini-Beispiel aus:

In [ ]:
SELECT order_id, EXPLODE(items) AS item
FROM VALUES
    ('A-1', ARRAY('Latte', 'Croissant')),
    ('A-2', ARRAY('Espresso'))
    AS t(order_id, items);

2. Sieh dir die rohen Web-Bestellungen an. `payment` und `shipping` sind **Structs** (auf ihre Felder greifst du mit einem Punkt zu, z. B. `shipping.city`), `items` ist ein **Array von Structs**:

In [ ]:
SELECT order_id, status, payment.method AS payment_method, shipping.city AS ship_to_city, items
FROM bronze.web_orders
LIMIT 10;

3. Löse die Artikel in eine Zeile pro Produkt auf, mit Positionsnummer:

In [ ]:
WITH web_items AS (
    SELECT order_id, POSEXPLODE(items) AS (item_pos, item)
    FROM bronze.web_orders
)
SELECT
    order_id,
    item_pos + 1      AS line_no,
    item.sku          AS sku,
    item.qty          AS quantity,
    item.unit_price   AS unit_price_eur
FROM web_items
ORDER BY order_id, line_no
LIMIT 20;

> **📝 Hinweis:** `EXPLODE` verwirft Zeilen, deren Array leer oder `NULL` ist. Wenn du sie behalten willst, nimm `EXPLODE_OUTER`. Die Silver-Tabelle `sales_order_line` nutzt `POSEXPLODE` genau wie oben – sieh dir `bundle/src/transformations/silver/sales_order_line.sql` an.

4. 🏋️ **Du bist dran:** Wie viele **Einheiten pro SKU** wurden online bestellt? Arbeite direkt auf dem rohen JSON in `bronze.web_orders` (ignoriere stornierte Bestellungen) und sortiere nach Einheiten.

In [ ]:
-- 🏋️ Du bist dran: schreibe hier deine Abfrage und führe die Zelle aus

> **💡 Lösung:** Hängst du fest oder bist du fertig? Vergleiche mit dem Notebook **SQL Lösungen** in diesem Ordner → *SQL 3 › Schritt 1 › Aufgabe 4*.

**Schritt 2: Die richtige Zeile pro Gruppe behalten mit QUALIFY**

`QUALIFY` filtert Zeilen anhand des Ergebnisses einer **Window-Funktion** – so wie `HAVING` nach Aggregaten filtert. Der häufigste Einsatz: genau eine Zeile pro Schlüssel behalten, z. B. die **neueste Version** eines Datensatzes:

```text
QUALIFY ROW_NUMBER() OVER (PARTITION BY <key> ORDER BY <newest first>) = 1
```

1. Führe das Mini-Beispiel aus. Anna hat zwei Versionen – welche bleibt übrig?

In [ ]:
SELECT *
FROM VALUES
    (1, 'Anna', 'Hamburg',  DATE'2024-01-10'),
    (1, 'Anna', 'Lüneburg', DATE'2025-06-01'),
    (2, 'Ben',  'Berlin',   DATE'2024-03-05')
    AS t(customer_id, name, city, updated_at)
QUALIFY ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY updated_at DESC) = 1;

2. Erinnerst du dich an die CRM-Historienzeilen aus SQL 1? Hier siehst du, warum sie gefährlich sind. Zähle die Kundenkarten einmal über den **rohen** CRM-Export und einmal direkt:

In [ ]:
SELECT
    (SELECT COUNT(*) FROM bronze.crm_loyalty_cards)                AS cards,
    (SELECT COUNT(*)
     FROM bronze.crm_customers AS c
     INNER JOIN bronze.crm_loyalty_cards AS lc
         ON c.customer_id = lc.customer_id)                        AS cards_after_join_with_raw_crm;

> **⚠️ Achtung:** Die zweite Zahl ist zu hoch: Jeder Kunde mit zwei CRM-Versionen verdoppelt seine Karten (**Join-Fan-out**, also Zeilenvervielfachung). Passiert das in einer Faktentabelle, wird der Umsatz aufgebläht, ohne dass es jemand merkt. Entferne Duplikate **bevor** du joinst.

3. Behalte nur die neueste Version jedes CRM-Kunden:

In [ ]:
SELECT customer_id, first_name, last_name, email, city, updated_at
FROM bronze.crm_customers
QUALIFY ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY updated_at DESC) = 1
ORDER BY customer_id;

4. 🏋️ **Du bist dran:** In SQL 2 hast du die Produkte pro Filiale mit einer CTE gerankt. Zeige jetzt nur die **Top 3 Produkte pro Filiale** nach Bruttoumsatz – in einer einzigen Abfrage mit `QUALIFY` (nutze `gold.fact_coffee_sales`, `gold.dim_store` und `gold.dim_product`).

In [ ]:
-- 🏋️ Du bist dran: schreibe hier deine Abfrage und führe die Zelle aus

> **💡 Lösung:** Hängst du fest oder bist du fertig? Vergleiche mit dem Notebook **SQL Lösungen** in diesem Ordner → *SQL 3 › Schritt 2 › Aufgabe 4*.

**Schritt 3: Fehlende Werte ersetzen mit COALESCE**

`COALESCE(a, b, c, …)` liefert den **ersten Wert, der nicht `NULL` ist**. Damit füllst du Lücken mit einem Standardwert oder nimmst einen Wert aus der ersten Quelle, die ihn hat.

1. Führe das Mini-Beispiel aus:

In [ ]:
SELECT
    COALESCE(NULL, NULL, 'third')   AS first_non_null,
    COALESCE(NULL, 0)               AS default_zero,
    COALESCE('Hamburg', 'Unknown')  AS keeps_value;

2. In den Kassenpositionen ist `discount_amount` leer, wenn kein Rabatt gewährt wurde. Aggregate **ignorieren** `NULL`-Werte einfach – und das ändert die Bedeutung eines Durchschnitts:

In [ ]:
SELECT
    AVG(discount_amount)              AS avg_discount_of_discounted_lines,
    AVG(COALESCE(discount_amount, 0)) AS avg_discount_of_all_lines
FROM bronze.pos_transaction_lines;

3. Wer hat bestellt? Bei Café-Bestellungen kennen wir den Kunden über die **Kundenkarte**, bei Online-Bestellungen über das **Kundenkonto im Onlineshop** – und bei Gästen wissen wir nichts. `COALESCE` nimmt den ersten gefundenen Kunden und verwendet `-1` („Guest“) als Ersatzwert. Genau so füllt die Gold-Faktentabelle `customer_key`:

In [ ]:
SELECT
    o.order_id,
    o.sales_channel,
    lc.customer_id                                AS customer_via_card,
    wc.customer_id                                AS customer_via_web_account,
    COALESCE(lc.customer_id, wc.customer_id, -1)  AS customer_key
FROM silver.sales_order AS o
LEFT JOIN silver.loyalty_card AS lc
    ON o.loyalty_card_no = lc.card_no
LEFT JOIN silver.customer AS wc
    ON o.web_account_id = wc.web_account_id
LIMIT 50;

> **✨ Tipp:** Einen fehlenden Dimensionsschlüssel durch einen festen „Unknown Member“ (unbekannten Kunden, hier `-1`) zu ersetzen, ist Best Practice im Gold Layer: Jede Faktenzeile findet dann einen Partner in der Dimension, und Reports zeigen einen klaren Eintrag „Guest“, statt bei einem INNER JOIN stillschweigend Zeilen zu verlieren.

4. 🏋️ **Du bist dran:** Online-Bestellungen haben eine Lieferstadt (`ship_to_city`), Café-Bestellungen nicht. Zähle die Bestellungen pro **„Bestellstadt“**: bei Online-Bestellungen die Lieferstadt, sonst die Stadt der Filiale (joine `silver.store`).

In [ ]:
-- 🏋️ Du bist dran: schreibe hier deine Abfrage und führe die Zelle aus

> **💡 Lösung:** Hängst du fest oder bist du fertig? Vergleiche mit dem Notebook **SQL Lösungen** in diesem Ordner → *SQL 3 › Schritt 3 › Aufgabe 4*.

**Schritt 4: Werte ableiten mit CASE und IFF**

`CASE` prüft Bedingungen von oben nach unten und liefert das Ergebnis der **ersten** Bedingung, die zutrifft (oder den `ELSE`-Wert). `IFF(condition, value_if_true, value_if_false)` ist eine Kurzform für genau zwei mögliche Ergebnisse.

1. Führe das Mini-Beispiel aus:

In [ ]:
SELECT
    price_eur,
    CASE
        WHEN price_eur < 3  THEN 'cheap'
        WHEN price_eur < 10 THEN 'regular'
        ELSE 'premium'
    END                                  AS price_band,
    IFF(price_eur >= 10, 'yes', 'no')    AS is_premium
FROM VALUES (2.60), (4.20), (32.00) AS t(price_eur);

2. Teile die Bestellungen in Klassen nach **Warenkorbgröße** ein und beschrifte den Kanal mit `IFF`:

In [ ]:
WITH order_totals AS (
    SELECT o.order_id, s.is_online, SUM(l.quantity * l.unit_price_eur - l.discount_eur) AS order_value_eur
    FROM silver.sales_order AS o
    INNER JOIN silver.sales_order_line AS l ON o.order_id = l.order_id
    INNER JOIN silver.store AS s ON o.store_id = s.store_id
    GROUP BY o.order_id, s.is_online
)
SELECT
    IFF(is_online, 'Online shop', 'Café') AS channel,
    CASE
        WHEN order_value_eur < 10 THEN '1 - Under €10'
        WHEN order_value_eur < 25 THEN '2 - €10–€25'
        WHEN order_value_eur < 50 THEN '3 - €25–€50'
        ELSE '4 - €50+'
    END                                   AS basket_size,
    COUNT(*)                              AS orders
FROM order_totals
GROUP BY ALL
ORDER BY channel, basket_size;

3. 🏋️ **Du bist dran:** Zähle die Kunden in `silver.customer` pro **Altersgruppe**: `Under 30`, `30–49`, `50 and older` und `Unknown` (reine Onlineshop-Kunden haben kein Geburtsdatum). Berechne das Alter als `YEAR(current_date()) - YEAR(birth_date)`.

In [ ]:
-- 🏋️ Du bist dran: schreibe hier deine Abfrage und führe die Zelle aus

> **💡 Lösung:** Hängst du fest oder bist du fertig? Vergleiche mit dem Notebook **SQL Lösungen** in diesem Ordner → *SQL 3 › Schritt 4 › Aufgabe 3*.

**Schritt 5: Codes übersetzen mit DECODE**

Quellsysteme lieben kurze Codes: `B`, `G`, `K`, `M` für Zahlungsarten, `S` und `R` für Steuerklassen. `DECODE(expr, search1, result1, search2, result2, …, default)` vergleicht einen Wert mit einer Liste von Codes und liefert das passende Ergebnis – eine kompakte Alternative zu einem `CASE` mit vielen Gleichheitsprüfungen.

1. Führe das Mini-Beispiel aus:

In [ ]:
SELECT
    tier_code,
    DECODE(tier_code, 'B', 'Bronze', 'S', 'Silver', 'G', 'Gold', 'Unknown') AS loyalty_tier
FROM VALUES ('G'), ('B'), ('X') AS t(tier_code);

2. Übersetze die Zahlungscodes der Café-Kassen und zähle die Bons:

In [ ]:
SELECT
    payment_code,
    DECODE(payment_code, 'B', 'Cash', 'G', 'Girocard', 'K', 'Credit Card', 'M', 'Mobile Payment', 'Unknown')
             AS payment_method,
    COUNT(*) AS receipts
FROM bronze.pos_transactions
GROUP BY ALL
ORDER BY receipts DESC;

3. Dasselbe mit `CASE` – vergleiche, wie viel länger das ist:

In [ ]:
SELECT
    payment_code,
    CASE payment_code
        WHEN 'B' THEN 'Cash'
        WHEN 'G' THEN 'Girocard'
        WHEN 'K' THEN 'Credit Card'
        WHEN 'M' THEN 'Mobile Payment'
        ELSE 'Unknown'
    END      AS payment_method,
    COUNT(*) AS receipts
FROM bronze.pos_transactions
GROUP BY ALL
ORDER BY receipts DESC;

4. 🏋️ **Du bist dran:** Mach die rohe ERP-Produktliste in `bronze.erp_products` lesbar: Übersetze `channel_code` (`S` = „Café only“, `O` = „Online only“, `B` = „Café and online“), `status_code` (`A` = „Active“, `D` = „Discontinued“) und `tax_class` (`S` = 0.19, `R` = 0.07 als Mehrwertsteuersatz).

In [ ]:
-- 🏋️ Du bist dran: schreibe hier deine Abfrage und führe die Zelle aus

> **💡 Lösung:** Hängst du fest oder bist du fertig? Vergleiche mit dem Notebook **SQL Lösungen** in diesem Ordner → *SQL 3 › Schritt 5 › Aufgabe 4*.

**Schritt 6: Den Wert am Maximum wählen mit MAX_BY**

`MAX(valid_from)` verrät dir, **wann** der neueste Preis begonnen hat – aber nicht, **wie hoch** er ist. `MAX_BY(value, ordering)` liefert den `value` der Zeile mit dem höchsten `ordering` innerhalb einer Gruppe. `MIN_BY` funktioniert umgekehrt. So brauchst du weder eine Window-Funktion noch einen Self-Join.

1. Führe das Mini-Beispiel aus:

In [ ]:
SELECT
    product,
    MAX(valid_from)            AS latest_change,
    MAX_BY(price, valid_from)  AS current_price,
    MIN_BY(price, valid_from)  AS first_price
FROM VALUES
    ('Latte', DATE'2023-01-01', 3.90),
    ('Latte', DATE'2025-08-01', 4.20),
    ('Espresso', DATE'2023-01-01', 2.40)
    AS t(product, valid_from, price)
GROUP BY product;

2. Der aktuelle Listenpreis jedes Produkts aus der rohen ERP-Preishistorie:

In [ ]:
SELECT
    product_id,
    MAX_BY(list_price_eur, valid_from)    AS current_list_price_eur,
    MAX_BY(cost_of_goods_eur, valid_from) AS current_cost_eur
FROM bronze.erp_product_prices
GROUP BY product_id
ORDER BY product_id;

3. Die **Lieblingsfiliale** jedes Kunden – die Filiale mit den meisten Bestellungen:

In [ ]:
WITH store_orders AS (
    SELECT customer_key, store_key, COUNT(DISTINCT order_id) AS orders
    FROM gold.fact_coffee_sales
    WHERE customer_key <> -1
    GROUP BY customer_key, store_key
)
SELECT customer_key, MAX_BY(store_key, orders) AS favourite_store_key
FROM store_orders
GROUP BY customer_key
LIMIT 20;

4. 🏋️ **Du bist dran:** Finde den **besten Tag** jeder Filiale: das Datum mit dem höchsten Bruttoumsatz und den Umsatz an diesem Tag.

In [ ]:
-- 🏋️ Du bist dran: schreibe hier deine Abfrage und führe die Zelle aus

> **💡 Lösung:** Hängst du fest oder bist du fertig? Vergleiche mit dem Notebook **SQL Lösungen** in diesem Ordner → *SQL 3 › Schritt 6 › Aufgabe 4*.

**Schritt 7: Bedingte Aggregate mit FILTER (WHERE …)**

Oft brauchst du dasselbe Aggregat für mehrere Teilmengen – z. B. Café-Umsatz **und** Online-Umsatz in einer Zeile. Statt mehrere Abfragen auszuführen, hängst du `FILTER (WHERE …)` an eine Aggregatfunktion: Nur die Zeilen, die die Bedingung erfüllen, werden aggregiert.

1. Führe das Mini-Beispiel aus:

In [ ]:
SELECT
    COUNT(*)                                    AS all_orders,
    COUNT(*) FILTER (WHERE channel = 'Online')  AS online_orders,
    SUM(amount) FILTER (WHERE channel = 'Café') AS cafe_revenue
FROM VALUES ('Café', 4.20), ('Online', 32.00), ('Café', 2.60) AS t(channel, amount);

2. Café- vs. Online-Umsatz pro Monat – in einem einzigen Durchgang über die Faktentabelle:

In [ ]:
SELECT
    DATE_TRUNC('MONTH', d.date)                                              AS month,
    SUM(f.gross_revenue_eur) FILTER (WHERE f.sales_channel = 'In-store')    AS cafe_revenue_eur,
    SUM(f.gross_revenue_eur) FILTER (WHERE f.sales_channel = 'Online')      AS online_revenue_eur,
    SUM(f.gross_revenue_eur)                                                 AS total_revenue_eur
FROM gold.fact_coffee_sales AS f
INNER JOIN gold.dim_date AS d ON f.date_key = d.date_key
GROUP BY DATE_TRUNC('MONTH', d.date)
ORDER BY month;

3. 🏋️ **Du bist dran:** Zähle für jedes **Café** die Bestellungen, die **bar** bezahlt wurden, und die **unbar** bezahlten (alle anderen Zahlungsarten), und berechne den Baranteil in Prozent.

In [ ]:
-- 🏋️ Du bist dran: schreibe hier deine Abfrage und führe die Zelle aus

> **💡 Lösung:** Hängst du fest oder bist du fertig? Vergleiche mit dem Notebook **SQL Lösungen** in diesem Ordner → *SQL 3 › Schritt 7 › Aufgabe 3*.

**Schritt 8: Mit Datumswerten rechnen – DATEADD und DATEDIFF**

- `DATEADD(unit, value, expr)` addiert Tage, Wochen, Monate … zu einem Datum oder Zeitstempel (mit einem negativen Wert wird subtrahiert). Das Ergebnis ist ein `TIMESTAMP` – umschließe es mit `CAST(… AS DATE)`, wenn du ein Datum brauchst.
- `DATEDIFF(unit, start, end)` liefert die Anzahl der Einheiten zwischen zwei Datumswerten.

1. Führe das Mini-Beispiel aus:

In [ ]:
SELECT
    DATEADD(DAY, -1, DATE'2026-01-01')                   AS new_years_eve,
    DATEADD(MONTH, 1, DATE'2026-01-31')                  AS one_month_later,
    CAST(DATEADD(YEAR, 1, DATE'2026-03-15') AS DATE)     AS one_year_later,
    DATEDIFF(DAY, DATE'2026-01-01', DATE'2026-12-24')    AS days_until_christmas_eve;

2. Das ERP speichert nur, wann ein Preis **beginnt**. Der Tag vor dem Start des nächsten Preises ist das **Ende** des aktuellen Preises – so berechnet `silver.product_price` die Spalte `valid_to`:

In [ ]:
SELECT
    product_id,
    valid_from,
    CAST(DATEADD(DAY, -1, LEAD(valid_from) OVER (PARTITION BY product_id ORDER BY valid_from)) AS DATE) AS valid_to,
    list_price_eur
FROM bronze.erp_product_prices
ORDER BY product_id, valid_from;

3. Welche Kunden haben schon lange nichts mehr gekauft? Berechne die Tage seit dem letzten Kauf und markiere Kunden, bei denen Abwanderung droht:

In [ ]:
SELECT
    customer_key,
    last_purchase_date,
    DATEDIFF(DAY, last_purchase_date, current_date())                       AS days_since_last_purchase,
    IFF(last_purchase_date < DATEADD(DAY, -90, current_date()), 'yes', 'no') AS churn_risk
FROM gold.dim_customer
WHERE last_purchase_date IS NOT NULL
ORDER BY days_since_last_purchase DESC
LIMIT 50;

4. 🏋️ **Du bist dran:** Neue Mitglieder im Treueprogramm erhalten im **ersten Jahr** ihrer Mitgliedschaft einen Willkommensbonus. Berechne für jede **aktive** Karte das Datum, an dem das erste Jahr endet, und zeige, ob das Mitglied heute noch im ersten Jahr ist.

In [ ]:
-- 🏋️ Du bist dran: schreibe hier deine Abfrage und führe die Zelle aus

> **💡 Lösung:** Hängst du fest oder bist du fertig? Vergleiche mit dem Notebook **SQL Lösungen** in diesem Ordner → *SQL 3 › Schritt 8 › Aufgabe 4*.